# Chapter 00: Fundamentals — Git, Pull Requests & GitHub Actions (Reference)

## Learning Objectives

- Walk a file through untracked -> staged -> committed and read git status at each step
- Perform the simplest possible merge (a fast-forward) and read the resulting log
- Recite the end-to-end pull-request lifecycle, branch to merge
- Identify the fields that actually mark a PR as successfully merged
- Parse a minimal GitHub Actions workflow file into its four basic pieces

## Setup

The next cell sets up reproducibility, the `PRA_MODE` toggle, and inserts the repo root onto `sys.path`. In this chapter, `PRA_MODE=live` doesn't mean network access -- it means "actually build a disposable local git repo and run real git commands," fully offline either way. You should see `PRA_MODE = 'fixture'` printed by default.

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

PRA_MODE = 'fixture'


## 1. The Three Areas: Untracked, Staged, Committed

The next cell prints the canned fixture transcript for a file moving through all three states. You should see `git status` describe the file as untracked, then staged, then report a clean working tree.

In [2]:
from labs.lab_00_fundamentals import FIXTURE_TRANSCRIPT

print(FIXTURE_TRANSCRIPT["status_untracked"])
print(FIXTURE_TRANSCRIPT["status_staged"])
print(FIXTURE_TRANSCRIPT["status_clean"])

On branch main
No commits yet
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.md

On branch main
No commits yet
Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   README.md

On branch main
nothing to commit, working tree clean



## 2. Live Mode: A Real Disposable Repo

The next cell only builds a real repo in `PRA_MODE=live`: it walks the three areas, diffs an edit, makes a second commit, then performs a fast-forward merge. In fixture mode it explains what would happen instead.

In [3]:
if PRA_MODE == "live":
    import tempfile
    from labs.lab_00_fundamentals import (
        init_demo_repo,
        demonstrate_three_areas,
        demonstrate_diff_and_second_commit,
        demonstrate_branch_and_merge,
    )

    with tempfile.TemporaryDirectory(prefix="pra_lab00_nb_") as tmp:
        repo = init_demo_repo(Path(tmp))
        areas = demonstrate_three_areas(repo)
        print(areas["clean"])
        result = demonstrate_diff_and_second_commit(repo)
        print(result["log"])
        print(demonstrate_branch_and_merge(repo))
else:
    print("Fixture mode: re-run with PRA_MODE=live to build a real disposable repo.")

Fixture mode: re-run with PRA_MODE=live to build a real disposable repo.


## 3. The Pull Request Lifecycle

The next cell prints the nine-stage PR lifecycle. You should see it go from "Branch created" through "Branch cleaned up".

In [4]:
from labs.lab_00_fundamentals import PR_LIFECYCLE_STAGES

for i, item in enumerate(PR_LIFECYCLE_STAGES, start=1):
    print(f"[{i}] {item['stage']}")
    print(f"    {item['what_happens']}")

[1] Branch created
    A new branch is created off main, e.g. `git checkout -b fix/readme-typo`.
[2] Commits made locally
    The add -> commit cycle from Steps 1-2 above, on that branch.
[3] Branch pushed
    `git push origin fix/readme-typo` uploads the branch's commits to GitHub.
[4] PR opened
    `gh pr create` (or the GitHub UI) asks GitHub to compare the branch against main and open a PR.
[5] Checks run
    GitHub Actions workflows fire automatically (Step 6 below, Chapter 08-09) and report pass/fail.
[6] Review requested/given
    A human (or nobody, if none is required) reviews the diff and approves or requests changes.
[7] Merge becomes available
    Once every required check passes and required reviews are satisfied, GitHub enables the merge button (or native auto-merge, Chapter 06, fires on its own).
[8] Merged
    GitHub combines the branch into main using whichever strategy is configured (Chapter 03) and marks the PR merged: true.
[9] Branch cleaned up
    The now-merged b

## 4. What a Successful Merge Looks Like

The next cell fetches a raw "successfully merged" PR object and extracts the fields that actually mark success. You should see `merged: True` alongside a real `merge_commit_sha` -- not just `state: closed`.

In [5]:
from labs.lab_00_fundamentals import fetch_merged_pr_example, describe_successful_merge

merged_pr = fetch_merged_pr_example()
summary = describe_successful_merge(merged_pr)
for key, value in summary.items():
    print(f"{key}: {value}")

state: closed
merged: True
merge_commit_sha: 7c4a9e8d13ad1e0c9a18bd5e2f4b6789012cdef3
merged_at: 2026-08-15T14:32:07Z


## 5. A Minimal Workflow File

The next cell parses `MINIMAL_WORKFLOW_YAML` into its four basic pieces. You should see the workflow's name, its `pull_request` trigger, and its one job's two steps.

In [6]:
from labs.lab_00_fundamentals import MINIMAL_WORKFLOW_YAML, describe_minimal_workflow

workflow = describe_minimal_workflow(MINIMAL_WORKFLOW_YAML)
print(f"name: {workflow['name']}")
print(f"trigger: {workflow['trigger']}")
print(
    f"job '{workflow['job_id']}' runs-on={workflow['runs_on']}: {workflow['step_names']}"
)

name: CI
trigger: ['pull_request']
job 'test' runs-on=ubuntu-latest: ['Checkout', 'Run tests']


## Takeaways & Next Steps

This notebook's takeaway is the full arc above: three areas, a fast-forward merge, the nine-stage PR lifecycle, what success looks like in the data, and the workflow file that makes checks run in the first place.

In [7]:
print("Re-run this notebook with PRA_MODE=live to build a real disposable repo.")

Re-run this notebook with PRA_MODE=live to build a real disposable repo.


---

📖 **Reading companion:** [Chapter 00: Fundamentals — Git, Pull Requests & GitHub Actions](../learning_modules/chapter_00_fundamentals.md)
🔬 **Try it live:** Re-run this notebook with `PRA_MODE=live PRA_REPO=<you>/practice_git_intro_to_pr_automerge`
